# Sleep Duration Analysis for Ages 18–24

This notebook analyses sleep duration data for people aged **18–24** using official survey data from the ABS source file.

## Goals
- extract the sleep-duration data for ages 18–24
- clean and wrangle the raw Excel table
- separate the results into **Weeknight**, **Weekend night**, and **Overall night**
- visualise the distribution of sleep-duration categories
- estimate the average sleep duration for each sleep type

## Why this matters
Sleep duration is an important factor in mental health, focus, energy, and recovery.  
This analysis helps compare how sleep patterns differ between weekdays and weekends for young adults.


In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import re


## Step 1: Load the raw Excel data

The Excel file contains a formatted table rather than a clean dataset.  
So the first step is to load the worksheet exactly as it appears and inspect the raw rows.


In [32]:
file_path = r"C:/Users/heryn/OneDrive/Documents/Semester 1 2026/FIT 5120/iteration 1 brainboost/project brain health/Measured-physical-activity-and-sleep-all/MPASDC02.xlsx"

df = pd.read_excel(file_path, sheet_name="Table 1.1_Proportions", header=None)

print(df.head(20))


                                                   0   \
0   This tab contains table 1.1, proportion of per...   
1                     Australian Bureau of Statistics   
2   Table 1.1 Average hours of sleep per night(a),...   
3   National Nutrition and Physical Activity Surve...   
4                                                 NaN   
5                                                 NaN   
6                                                 NaN   
7             Average hours of sleep per weeknight(e)   
8                                   Less than 6 hours   
9                              6 to less than 7 hours   
10                             7 to less than 8 hours   
11                             8 to less than 9 hours   
12                            9 to less than 10 hours   
13                                   10 hours or more   
14                                      Total persons   
15        Average hours of sleep per weekend night(f)   
16                             

## Step 2: Extract the sleep table for ages 18–24

The worksheet includes multiple age groups and formatting rows.  
This step:
- uses the correct row as the column header
- keeps only the table body
- selects the **18–24** age group
- renames the selected value column to `value`


In [33]:
# Use row 5 as the column-header row
columns = ["category"] + df.iloc[5, 1:].tolist()

# Keep the main body of the table
table = df.iloc[7:38, 0:len(columns)].copy()
table.columns = columns

# Clean column names
table.columns = [str(col).strip() for col in table.columns]

# Clean category labels
table["category"] = table["category"].astype(str).str.strip()

# Keep only ages 18–24
sleep_18_24 = table[["category", "18–24"]].copy()
sleep_18_24 = sleep_18_24.rename(columns={"18–24": "value"})

print(sleep_18_24.head(20))


                                       category value
7       Average hours of sleep per weeknight(e)   NaN
8                             Less than 6 hours  10.3
9                        6 to less than 7 hours  18.9
10                       7 to less than 8 hours  33.3
11                       8 to less than 9 hours  27.3
12                      9 to less than 10 hours   9.8
13                             10 hours or more   0.4
14                                Total persons   100
15  Average hours of sleep per weekend night(f)   NaN
16                            Less than 6 hours  17.7
17                       6 to less than 7 hours     9
18                       7 to less than 8 hours  25.7
19                       8 to less than 9 hours  25.7
20                      9 to less than 10 hours   9.7
21                             10 hours or more  12.2
22                                Total persons   100
23             Average hours of sleep per night   NaN
24                          

## Step 3: Clean and wrangle the extracted data

The extracted column still contains:
- notes
- footnotes
- non-data rows
- values stored as text

This step removes irrelevant rows and converts the percentages into numeric values.


In [34]:
# Remove blank rows
sleep_18_24 = sleep_18_24.dropna(subset=["category"])

# Remove notes / footnotes / copyright rows
sleep_18_24 = sleep_18_24[
    ~sleep_18_24["category"].str.contains(
        "Footnotes|Commonwealth of Australia|Proportions may not add up|^\(",
        case=False,
        na=False,
        regex=True
    )
].copy()

# Clean the value column
sleep_18_24["value"] = (
    sleep_18_24["value"]
    .astype(str)
    .str.replace("#", "", regex=False)
    .str.replace(",", ".", regex=False)
)

# Convert to numeric
sleep_18_24["value"] = pd.to_numeric(sleep_18_24["value"], errors="coerce")

print(sleep_18_24)


                                             category  value
7             Average hours of sleep per weeknight(e)    NaN
8                                   Less than 6 hours   10.3
9                              6 to less than 7 hours   18.9
10                             7 to less than 8 hours   33.3
11                             8 to less than 9 hours   27.3
12                            9 to less than 10 hours    9.8
13                                   10 hours or more    0.4
14                                      Total persons  100.0
15        Average hours of sleep per weekend night(f)    NaN
16                                  Less than 6 hours   17.7
17                             6 to less than 7 hours    9.0
18                             7 to less than 8 hours   25.7
19                             8 to less than 9 hours   25.7
20                            9 to less than 10 hours    9.7
21                                   10 hours or more   12.2
22                      

<>:7: SyntaxWarning:

invalid escape sequence '\('

<>:7: SyntaxWarning:

invalid escape sequence '\('

C:\Users\heryn\AppData\Local\Temp\ipykernel_58120\492998915.py:7: SyntaxWarning:

invalid escape sequence '\('



## Step 4: Separate the sleep data into 3 groups

The cleaned table contains 3 sleep sections:
- **Weeknight**
- **Weekend night**
- **Overall night**

Each section also includes a heading row and a `Total persons` row, so the rows must be sliced carefully.

This is an important wrangling step because incorrect slicing will assign categories to the wrong sleep type.


In [35]:
sleep_rows = sleep_18_24.reset_index(drop=True)

# Correct row positions after cleaning:
# 0  = weeknight heading
# 1:7 = weeknight categories
# 7  = total persons
# 8  = weekend heading
# 9:15 = weekend categories
# 15 = total persons
# 16 = overall heading
# 17:23 = overall categories
# 23 = total persons

weeknight_sleep = sleep_rows.iloc[1:7].copy()
weekend_sleep = sleep_rows.iloc[9:15].copy()
overall_sleep = sleep_rows.iloc[17:23].copy()

weeknight_sleep["sleep_type"] = "Weeknight"
weekend_sleep["sleep_type"] = "Weekend night"
overall_sleep["sleep_type"] = "Overall night"

sleep_combined = pd.concat(
    [weeknight_sleep, weekend_sleep, overall_sleep],
    ignore_index=True
)

print(sleep_combined)


                   category  value     sleep_type
0         Less than 6 hours   10.3      Weeknight
1    6 to less than 7 hours   18.9      Weeknight
2    7 to less than 8 hours   33.3      Weeknight
3    8 to less than 9 hours   27.3      Weeknight
4   9 to less than 10 hours    9.8      Weeknight
5          10 hours or more    0.4      Weeknight
6         Less than 6 hours   17.7  Weekend night
7    6 to less than 7 hours    9.0  Weekend night
8    7 to less than 8 hours   25.7  Weekend night
9    8 to less than 9 hours   25.7  Weekend night
10  9 to less than 10 hours    9.7  Weekend night
11         10 hours or more   12.2  Weekend night
12        Less than 6 hours    9.5  Overall night
13   6 to less than 7 hours   18.0  Overall night
14   7 to less than 8 hours   31.0  Overall night
15   8 to less than 9 hours   35.8  Overall night
16  9 to less than 10 hours    5.1  Overall night
17         10 hours or more    0.5  Overall night


## Step 5: Order the sleep categories

To make the visualisations easier to read, the sleep-duration categories are ordered from the shortest duration to the longest duration.


In [36]:
category_order = [
    "Less than 6 hours",
    "6 to less than 7 hours",
    "7 to less than 8 hours",
    "8 to less than 9 hours",
    "9 to less than 10 hours",
    "10 hours or more"
]

sleep_combined["category"] = pd.Categorical(
    sleep_combined["category"],
    categories=category_order,
    ordered=True
)

sleep_combined = sleep_combined.sort_values(["sleep_type", "category"]).reset_index(drop=True)

print(sleep_combined)


                   category  value     sleep_type
0         Less than 6 hours    9.5  Overall night
1    6 to less than 7 hours   18.0  Overall night
2    7 to less than 8 hours   31.0  Overall night
3    8 to less than 9 hours   35.8  Overall night
4   9 to less than 10 hours    5.1  Overall night
5          10 hours or more    0.5  Overall night
6         Less than 6 hours   17.7  Weekend night
7    6 to less than 7 hours    9.0  Weekend night
8    7 to less than 8 hours   25.7  Weekend night
9    8 to less than 9 hours   25.7  Weekend night
10  9 to less than 10 hours    9.7  Weekend night
11         10 hours or more   12.2  Weekend night
12        Less than 6 hours   10.3      Weeknight
13   6 to less than 7 hours   18.9      Weeknight
14   7 to less than 8 hours   33.3      Weeknight
15   8 to less than 9 hours   27.3      Weeknight
16  9 to less than 10 hours    9.8      Weeknight
17         10 hours or more    0.4      Weeknight


## Step 6: Visualise the sleep-duration distribution

This grouped bar chart compares the percentage of 18-24-year-olds in each sleep-duration category across:
- weeknights
- weekend nights
- overall nights

It also includes the **average sleep duration** for each sleep type inside the chart so the distribution and the summary can be read together.


In [37]:
sleep_combined["value"] = pd.to_numeric(sleep_combined["value"], errors="coerce")
sleep_combined["midpoint_hours"] = sleep_combined["category"].astype(str).apply(estimate_midpoint)
sleep_combined["midpoint_hours"] = pd.to_numeric(sleep_combined["midpoint_hours"], errors="coerce")

avg_sleep = (
    sleep_combined.groupby("sleep_type", as_index=False)
    .agg(
        avg_hours=(
            "midpoint_hours",
            lambda s: (s * sleep_combined.loc[s.index, "value"]).sum()
            / sleep_combined.loc[s.index, "value"].sum()
        )
    )
)

avg_lookup = dict(zip(avg_sleep["sleep_type"], avg_sleep["avg_hours"].round(2)))
avg_text = (
    f"Weeknight avg: {avg_lookup['Weeknight']:.2f} h<br>"
    f"Weekend night avg: {avg_lookup['Weekend night']:.2f} h<br>"
    f"Overall night avg: {avg_lookup['Overall night']:.2f} h"
)

fig = px.bar(
    sleep_combined,
    x="category",
    y="value",
    color="sleep_type",
    barmode="group",
    text="value",
    title="Sleep Duration Distribution for Ages 18-24",
    labels={
        "category": "Sleep duration",
        "value": "Percentage",
        "sleep_type": "Sleep type"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    hovertemplate=(
        "Sleep type: %{fullData.name}<br>"
        "Duration: %{x}<br>"
        "Percentage: %{y:.1f}%<extra></extra>"
    )
)

fig.add_annotation(
    x=0.99,
    y=0.98,
    xref="paper",
    yref="paper",
    xanchor="right",
    yanchor="top",
    align="left",
    showarrow=False,
    text=avg_text,
    bgcolor="rgba(255,255,255,0.92)",
    bordercolor="#D9E2F2",
    borderwidth=1,
    font=dict(size=12)
)

fig.update_layout(
    template="plotly_white",
    height=650,
    title_x=0.5,
    xaxis_tickangle=-25
)

fig.show()


## Step 7: Estimate midpoint hours for each sleep band

The ABS data is grouped into categorical ranges, such as:
- less than 6 hours
- 6 to less than 7 hours
- 10 hours or more

To estimate the average sleep duration, each category is converted into a midpoint value.
For example:
- `6 to less than 7 hours` becomes `6.5`
- `Less than 6 hours` becomes `5.5`
- `10 hours or more` becomes `10.5`


In [38]:
def estimate_midpoint(label: str) -> float:
    text = str(label).strip().lower()

    m = re.search(r"(\d+(?:\.\d+)?)\s+to\s+less\s+than\s+(\d+(?:\.\d+)?)", text)
    if m:
        low = float(m.group(1))
        high = float(m.group(2))
        return (low + high) / 2

    m = re.search(r"less than\s+(\d+(?:\.\d+)?)", text)
    if m:
        high = float(m.group(1))
        return high - 0.5

    m = re.search(r"(\d+(?:\.\d+)?)\s+hours?\s+or\s+more", text)
    if m:
        low = float(m.group(1))
        return low + 0.5

    return None

sleep_combined["midpoint_hours"] = sleep_combined["category"].apply(estimate_midpoint)

print(sleep_combined[["category", "midpoint_hours"]].drop_duplicates())


                  category midpoint_hours
0        Less than 6 hours            5.5
1   6 to less than 7 hours            6.5
2   7 to less than 8 hours            7.5
3   8 to less than 9 hours            8.5
4  9 to less than 10 hours            9.5
5         10 hours or more           10.5


## Step 8: Review the weighted average sleep duration

Because the data is stored as percentages in categories rather than individual observations, the average sleep duration is estimated using a weighted average.

### Formula
Estimated average sleep hours =  
sum of (`midpoint hours * percentage`) divided by sum of percentages

The same weighted averages are already added to the main grouped chart above, and this step prints the values as a summary table.


In [39]:
def estimate_midpoint(label: str) -> float:
    text = str(label).strip().lower()

    m = re.search(r"(\d+(?:\.\d+)?)\s+to\s+less\s+than\s+(\d+(?:\.\d+)?)", text)
    if m:
        low = float(m.group(1))
        high = float(m.group(2))
        return (low + high) / 2

    m = re.search(r"less than\s+(\d+(?:\.\d+)?)", text)
    if m:
        high = float(m.group(1))
        return high - 0.5

    m = re.search(r"(\d+(?:\.\d+)?)\s+hours?\s+or\s+more", text)
    if m:
        low = float(m.group(1))
        return low + 0.5

    return None

sleep_combined["value"] = pd.to_numeric(sleep_combined["value"], errors="coerce")
sleep_combined["midpoint_hours"] = sleep_combined["category"].astype(str).apply(estimate_midpoint)
sleep_combined["midpoint_hours"] = pd.to_numeric(sleep_combined["midpoint_hours"], errors="coerce")

print(sleep_combined[["category", "midpoint_hours"]].drop_duplicates())


                  category  midpoint_hours
0        Less than 6 hours             5.5
1   6 to less than 7 hours             6.5
2   7 to less than 8 hours             7.5
3   8 to less than 9 hours             8.5
4  9 to less than 10 hours             9.5
5         10 hours or more            10.5


In [40]:
if 'avg_sleep' not in globals():
    avg_sleep = (
        sleep_combined.groupby("sleep_type", as_index=False)
        .agg(
            avg_hours=(
                "midpoint_hours",
                lambda s: (s * sleep_combined.loc[s.index, "value"]).sum()
                / sleep_combined.loc[s.index, "value"].sum()
            )
        )
    )

avg_sleep["avg_hours"] = avg_sleep["avg_hours"].round(2)

print(avg_sleep)


      sleep_type  avg_hours
0  Overall night   7.605105
1  Weekend night   7.873000
2      Weeknight   7.586000


## Step 9: Visualise the sleep-duration trend with average markers

This line chart shows the sleep-duration distribution across estimated hours and overlays the **average sleep duration** for each sleep type.

The dashed vertical lines mark the weighted-average sleep duration for:
- **Weeknight**
- **Weekend night**
- **Overall night**

This makes the chart easier to present because both the distribution and the average are visible in the same figure.


In [41]:
sleep_combined["value"] = pd.to_numeric(sleep_combined["value"], errors="coerce")
sleep_combined["midpoint_hours"] = sleep_combined["category"].astype(str).apply(estimate_midpoint)
sleep_combined["midpoint_hours"] = pd.to_numeric(sleep_combined["midpoint_hours"], errors="coerce")

avg_sleep_chart = (
    sleep_combined.groupby("sleep_type", as_index=False)
    .agg(
        avg_hours=(
            "midpoint_hours",
            lambda s: (s * sleep_combined.loc[s.index, "value"]).sum()
            / sleep_combined.loc[s.index, "value"].sum()
        )
    )
)

avg_lookup = dict(zip(avg_sleep_chart["sleep_type"], avg_sleep_chart["avg_hours"]))
summary_text = (
    f"Weeknight: {avg_lookup['Weeknight']:.2f} h | "
    f"Weekend night: {avg_lookup['Weekend night']:.2f} h | "
    f"Overall night: {avg_lookup['Overall night']:.2f} h"
)

fig = px.line(
    sleep_combined,
    x="midpoint_hours",
    y="value",
    color="sleep_type",
    markers=True,
    title="Sleep Duration Distribution for Ages 18-24",
    labels={
        "midpoint_hours": "Estimated sleep hours",
        "value": "Percentage",
        "sleep_type": "Sleep type"
    }
)

fig.update_traces(
    mode="lines+markers",
    hovertemplate=(
        "Sleep type: %{fullData.name}<br>"
        "Estimated hours: %{x:.1f}<br>"
        "Percentage: %{y:.1f}%<extra></extra>"
    )
)

for _, row in avg_sleep_chart.iterrows():
    fig.add_vline(
        x=row["avg_hours"],
        line_dash="dash",
        line_color="#2F436E",
        line_width=2,
        opacity=0.9,
    )

fig.add_annotation(
    x=0.5,
    y=1.10,
    xref="paper",
    yref="paper",
    text=summary_text,
    showarrow=False,
    font=dict(size=14, color="#2F436E"),
    align="center"
)

fig.update_layout(
    template="plotly_white",
    height=620,
    title_x=0.5,
    margin=dict(t=120, r=40, b=60, l=60),
    xaxis=dict(
        tickmode="array",
        tickvals=[5.5, 6.5, 7.5, 8.5, 9.5, 10.5],
        ticktext=["<6 h", "6-<7 h", "7-<8 h", "8-<9 h", "9-<10 h", "10+ h"]
    ),
    yaxis_title="Percentage",
    legend_title_text="Sleep type",
)

fig.show()


## Step 10: Key findings

From the cleaned and wrangled data:

- **Weeknight sleep** is slightly lower than **weekend-night sleep**
- the largest proportion of 18–24-year-olds falls into the **7 to less than 9 hours** range
- the estimated averages suggest that weekend sleep is higher than weeknight sleep
- the overall-night pattern provides a combined summary of the full distribution

This supports the idea that young adults tend to sleep longer on weekends than on weeknights.
